In [1]:
import os
import sqlite3

from rdkit import Chem, RDLogger
from tqdm.notebook import tqdm

logger = RDLogger.logger()
logger.setLevel(RDLogger.CRITICAL)

FUNCTIONAL_GROUPS = [
    Chem.MolFromSmarts("[OX2H]"),  # Hydroxyl
    Chem.MolFromSmarts("[#16X2H]"),  # Thiol
    Chem.MolFromSmarts("[CX3](=[OX1])O"),  # Ester, acid, etc.
    Chem.MolFromSmarts("[CX3H1](=O)[#6]"),  # -CHO
    Chem.MolFromSmarts("S(=O)(=O)"),  # -SO2R
    Chem.MolFromSmarts("[$([NX4+]),$([NX4]=*)]"),  # NR4+
]

from rdkit import Chem


def filter_pacs(mol: Chem.Mol) -> bool:
    """
    Checks if a molecule meets the PAC (Polycyclic Aromatic Compound) criteria.

    Criteria:
    1. No fragmented molecules.
    2. No molecules with radicals.
    3. Must be electrically neutral.
    4. Rings can only be 5 or 6-membered.
    5. At least 1 6-membered ring, and it must be aromatic.
    6. Allowed atoms: C, H, O, N, S, F, Cl.
    7. Filter specific functional groups (based on FUNCTIONAL_GROUPS list).
    8. No sp3 carbons with 4 non-hydrogen connections (quaternary sp3 carbons).
    9. No long-chain groups (> 4 carbons).
    10. Filter out bridged systems (shared atoms between rings > 2 or non-bonded shared atoms).
    11. Nitrogen must never be bonded to hydrogen (No N-H bonds).
    """
    if mol is None:
        return False

    if "." in Chem.MolToSmiles(mol):
        return False

    for atom in mol.GetAtoms():
        if atom.GetNumRadicalElectrons() > 0:
            return False

    if Chem.GetFormalCharge(mol) != 0:
        return False

    ring_info = mol.GetRingInfo().AtomRings()
    if not ring_info:
        return False

    has_aromatic_six_membered = False
    for ring in ring_info:
        r_len = len(ring)
        if r_len == 6:
            # Check if all atoms in this 6-membered ring are aromatic
            if all(mol.GetAtomWithIdx(idx).GetIsAromatic() for idx in ring):
                has_aromatic_six_membered = True
        elif r_len != 5:
            return False

    if not has_aromatic_six_membered:
        return False

    allowed_atoms = {6, 1, 8, 7, 16, 9, 17}
    for atom in mol.GetAtoms():
        if atom.GetAtomicNum() not in allowed_atoms:
            return False

    for fg in FUNCTIONAL_GROUPS:
        if mol.HasSubstructMatch(fg):
            return False

    if mol.HasSubstructMatch(Chem.MolFromSmarts("[CX4;H0]")):
        return False

    if mol.HasSubstructMatch(Chem.MolFromSmarts("[AR0]~[AR0]~[AR0]~[AR0]")):
        return False

    for i in range(len(ring_info)):
        for j in range(i + 1, len(ring_info)):
            r1, r2 = set(ring_info[i]), set(ring_info[j])
            intersect = r1.intersection(r2)
            if len(intersect) > 2:
                return False
            if len(intersect) == 2:
                idx1, idx2 = list(intersect)
                if not mol.GetBondBetweenAtoms(idx1, idx2):
                    return False

    if mol.HasSubstructMatch(Chem.MolFromSmarts("[N;!H0]")):
        return False

    return True

In [3]:
db_path = "../data/pac5.db"
os.makedirs(os.path.dirname(db_path), exist_ok=True)

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Create table
cursor.execute("""
    CREATE TABLE IF NOT EXISTS pac5 (
        pubchem_cid INTEGER PRIMARY KEY,
        smiles TEXT
    )
""")
conn.commit()

input_file = "../data/pubchem/CID-SMILES"

batch_size = 500
resume = 0
data_to_insert = []

with open(input_file, "r", encoding="utf-8") as f:
    for i, line in enumerate(tqdm(f, desc="Filtering PACs", total=178_243_495)):
        if i < resume:
            continue
        parts = line.strip().split("\t")
        cid, smiles = parts[0], parts[1]
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            continue
        try:
            if filter_pacs(mol):
                data_to_insert.append((int(cid), smiles))
        except Exception:
            continue
        # Periodically commit to DB
        if len(data_to_insert) >= batch_size:
            cursor.executemany(
                "INSERT OR IGNORE INTO pac5 VALUES (?, ?)", data_to_insert
            )
            conn.commit()
            data_to_insert = []

# Final commit for remaining items
if data_to_insert:
    cursor.executemany("INSERT OR IGNORE INTO pac5 VALUES (?, ?)", data_to_insert)
    conn.commit()

conn.close()

Filtering PACs:   0%|          | 0/178243495 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [1]:
import os
import sqlite3

from rdkit import Chem, RDLogger
from tqdm.notebook import tqdm

logger = RDLogger.logger()
logger.setLevel(RDLogger.CRITICAL)

db_path = "../data/pac5.db"

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

In [8]:
cursor.execute(
    "SELECT COUNT(*) FROM pac5 WHERE excluded = 0 AND has_characteristics_5 = 1"
)
total_pacs = cursor.fetchone()[0]
print(f"Total PACs in database: {total_pacs}")

Total PACs in database: 853399


In [10]:
import sqlite3

import pandas as pd
import plotly.express as px

db_path = "../data/pac5.db"
conn = sqlite3.connect(db_path)

query = """
SELECT
    has_characteristics_1,
    has_characteristics_2,
    has_characteristics_3,
    has_characteristics_4,
    has_characteristics_5
FROM pac5 WHERE selected = 1
"""
df = pd.read_sql_query(query, conn)
conn.close()

category_cols = [
    "has_characteristics_1",
    "has_characteristics_2",
    "has_characteristics_3",
    "has_characteristics_4",
    "has_characteristics_5",
]

df["combination_pattern"] = df[category_cols].astype(str).agg("-".join, axis=1)

combination_counts = df["combination_pattern"].value_counts().reset_index()
combination_counts.columns = [
    "Combination pattern (Categories 1 to 5)",
    "Molecule count",
]

# Remove baseline category (no characteristics present)
combination_counts = combination_counts[
    combination_counts["Combination pattern (Categories 1 to 5)"] != "0-0-0-0-0"
]

# Format labels with comma thousands separators
combination_counts["Label"] = combination_counts["Molecule count"].apply(
    lambda x: f"{x:,}"
)

fig = px.bar(
    combination_counts,
    x="Combination pattern (Categories 1 to 5)",
    y="Molecule count",
    text="Label",
    title="Frequency of category combinations for PACs",
    labels={
        "Combination pattern (Categories 1 to 5)": "Binary state (1=Present, 0=Absent)"
    },
    log_y=True,
    template="plotly_white",
    color="Molecule count",
    color_continuous_scale="Blues",
)

fig.update_traces(
    textposition="outside",
    textfont_size=11,
    marker_line_width=1,
    marker_line_color="black",
)

fig.update_layout(
    xaxis_tickangle=-45, coloraxis_showscale=False, margin=dict(t=50, b=50, l=50, r=50)
)

max_val = combination_counts["Molecule count"].max()
fig.update_yaxes(range=[0, len(str(max_val)) + 0.5])

fig.show()

In [11]:
import sqlite3

import pandas as pd
import plotly.express as px

db_path = "../data/pac5.db"
conn = sqlite3.connect(db_path)

query = """
SELECT
    has_characteristics_1,
    has_characteristics_2,
    has_characteristics_3,
    has_characteristics_4,
    has_characteristics_5,
    num_total_rings
FROM pac5 WHERE selected = 1
"""
df = pd.read_sql_query(query, conn)
conn.close()

category_cols = [
    "has_characteristics_1",
    "has_characteristics_2",
    "has_characteristics_3",
    "has_characteristics_4",
    "has_characteristics_5",
]

df["combination_pattern"] = df[category_cols].astype(str).agg("-".join, axis=1)

df_filtered = df[df["combination_pattern"] != "0-0-0-0-0"].copy()

df_filtered["num_total_rings"] = (
    df_filtered["num_total_rings"].fillna(-1).astype(int).astype(str)
)

pattern_totals = df_filtered["combination_pattern"].value_counts().index.tolist()

combination_counts = (
    df_filtered.groupby(["combination_pattern", "num_total_rings"])
    .size()
    .reset_index(name="Molecule count")
)

fig = px.bar(
    combination_counts,
    x="combination_pattern",
    y="Molecule count",
    color="num_total_rings",
    title="Frequency of category combinations for PACs by ring count",
    labels={
        "combination_pattern": "Binary state (1=Present, 0=Absent)",
        "num_total_rings": "Total rings",
    },
    log_y=False,
    template="plotly_white",
    category_orders={"combination_pattern": pattern_totals},
)

fig.update_layout(
    xaxis_tickangle=-45, margin=dict(t=50, b=50, l=50, r=50), barmode="stack"
)

fig.show()

In [17]:
import sqlite3

import pandas as pd
import plotly.express as px

db_path = "../data/pac5.db"
conn = sqlite3.connect(db_path)

query = """
SELECT num_total_rings
FROM pac5
WHERE selected = 1
"""
df = pd.read_sql_query(query, conn)
conn.close()

fig = px.histogram(
    df,
    x="num_total_rings",
    title="Distribution of total rings for PACs",
    labels={"num_total_rings": "Number of total rings"},
    template="plotly_white",
)

fig.update_layout(bargap=0.1, margin=dict(t=50, b=50, l=50, r=50))

fig.show()

In [2]:
fig.write_image("./a.svg")

In [2]:
import sqlite3

import numpy as np
import pandas as pd

db_path = "../data/pac5.db"
conn = sqlite3.connect(db_path)

query = """
SELECT
    pubchem_cid,
    has_characteristics_1,
    has_characteristics_2,
    has_characteristics_3,
    has_characteristics_4,
    has_characteristics_5
FROM pac5 WHERE in_scope = 1 AND excluded = 0
"""
df = pd.read_sql_query(query, conn)

category_cols = [
    "has_characteristics_1",
    "has_characteristics_2",
    "has_characteristics_3",
    "has_characteristics_4",
    "has_characteristics_5",
]

df["combination_pattern"] = df[category_cols].astype(str).agg("-".join, axis=1)

target_size = 200000
total_size = len(df)

# Target count per pattern, proportional to pattern frequency
counts = df["combination_pattern"].value_counts()
proportions = counts / total_size
target_counts = (proportions * target_size).round().astype(int)

# Fix rounding so counts sum to target_size
difference = target_size - target_counts.sum()
if difference != 0:
    largest_group = target_counts.idxmax()
    target_counts[largest_group] += difference

sampled_list = []
for pattern, count in target_counts.items():
    if count > 0:
        group_df = df[df["combination_pattern"] == pattern]
        sampled_group = group_df.sample(n=count, random_state=42)
        sampled_list.append(sampled_group)

sampled_df = (
    pd.concat(sampled_list).sample(frac=1, random_state=42).reset_index(drop=True)
)

# CIDs as list of 1-tuples for executemany()
selected_ids = [(int(cid),) for cid in sampled_df["pubchem_cid"]]

cursor = conn.cursor()

try:
    cursor.execute("ALTER TABLE pac5 ADD COLUMN selected INTEGER DEFAULT 0")
except sqlite3.OperationalError as e:
    if "duplicate column name" in str(e):
        print("Column 'selected' already exists. Resetting all to 0.")
        cursor.execute("UPDATE pac5 SET selected = 0")
    else:
        raise e

print(f"Updating {len(selected_ids)} rows")
cursor.executemany(
    """
    UPDATE pac5
    SET selected = 1
    WHERE pubchem_cid = ?
""",
    selected_ids,
)

conn.commit()
conn.close()

Column 'selected' already exists. Resetting all to 0.
Updating 200000 rows


In [11]:
from pac.convert import smiles_to_pac

mol = smiles_to_pac("O=c(cc[nH]1)[nH]c1=O")

In [10]:
import sqlite3

import pandas as pd

db_path = "../data/pac5.db"
conn = sqlite3.connect(db_path)

query = """
SELECT
    pubchem_cid,
    has_characteristics_1,
    has_characteristics_2,
    has_characteristics_3,
    has_characteristics_4,
    has_characteristics_5
FROM pac5 WHERE in_scope = 1 AND excluded = 0
"""
df = pd.read_sql_query(query, conn)

category_cols = [
    "has_characteristics_1",
    "has_characteristics_2",
    "has_characteristics_3",
    "has_characteristics_4",
    "has_characteristics_5",
]

df["combination_pattern"] = df[category_cols].astype(str).agg("-".join, axis=1)

# Remove baseline category from pool
df_filtered = df[df["combination_pattern"] != "0-0-0-0-0"]

target_size = 100000
counts = df_filtered["combination_pattern"].value_counts().to_dict()

available_patterns = list(counts.keys())
allocated = {p: 0 for p in available_patterns}
remaining_target = target_size

# Water-filling algorithm to equalize bars
while remaining_target > 0 and len(available_patterns) > 0:
    per_pattern_target = remaining_target // len(available_patterns)

    # Find groups with fewer molecules than current even split
    shortfall_patterns = [
        p for p in available_patterns if counts[p] <= per_pattern_target
    ]

    if shortfall_patterns:
        # Take 100% of these small groups
        for p in shortfall_patterns:
            allocated[p] = counts[p]
            remaining_target -= counts[p]
            available_patterns.remove(p)
    else:
        # Remaining groups are large enough for even split
        for p in available_patterns:
            allocated[p] = per_pattern_target
            remaining_target -= per_pattern_target

        # Distribute any leftover remainders from division rounding
        for p in available_patterns[:remaining_target]:
            allocated[p] += 1
        remaining_target = 0
        break

sampled_list = []
for pattern, count in allocated.items():
    if count > 0:
        group_df = df_filtered[df_filtered["combination_pattern"] == pattern]
        sampled_group = group_df.sample(n=count, random_state=42)
        sampled_list.append(sampled_group)

sampled_df = (
    pd.concat(sampled_list).sample(frac=1, random_state=42).reset_index(drop=True)
)

selected_ids = [(int(cid),) for cid in sampled_df["pubchem_cid"]]
cursor = conn.cursor()

try:
    cursor.execute("ALTER TABLE pac5 ADD COLUMN selected INTEGER DEFAULT 0")
except sqlite3.OperationalError as e:
    if "duplicate column name" in str(e):
        print("Column 'selected' already exists. Resetting all to 0.")
        cursor.execute("UPDATE pac5 SET selected = 0")
    else:
        raise e

print(f"Updating {len(selected_ids)} rows")
cursor.executemany(
    """
    UPDATE pac5
    SET selected = 1
    WHERE pubchem_cid = ?
""",
    selected_ids,
)

conn.commit()
conn.close()

print(f"Molecules selected: {len(sampled_df)}")

Column 'selected' already exists. Resetting all to 0.
Updating 100000 rows
Molecules selected: 100000


In [2]:
import sqlite3

import pandas as pd


def calculate_water_fill(counts_dict, target_size):
    available_keys = list(counts_dict.keys())
    allocated = {k: 0 for k in available_keys}
    remaining_target = target_size

    while remaining_target > 0 and len(available_keys) > 0:
        per_key_target = remaining_target // len(available_keys)
        shortfall_keys = [k for k in available_keys if counts_dict[k] <= per_key_target]

        if shortfall_keys:
            for k in shortfall_keys:
                allocated[k] = counts_dict[k]
                remaining_target -= counts_dict[k]
                available_keys.remove(k)
        else:
            for k in available_keys:
                allocated[k] = per_key_target
                remaining_target -= per_key_target

            for k in available_keys[:remaining_target]:
                allocated[k] += 1
            remaining_target = 0
            break

    return allocated


db_path = "../data/pac5.db"
conn = sqlite3.connect(db_path)

query = """
SELECT
    pubchem_cid,
    has_characteristics_1,
    has_characteristics_2,
    has_characteristics_3,
    has_characteristics_4,
    has_characteristics_5,
    num_total_rings
FROM pac5 WHERE in_scope = 1 AND excluded = 0
"""
df = pd.read_sql_query(query, conn)

category_cols = [
    "has_characteristics_1",
    "has_characteristics_2",
    "has_characteristics_3",
    "has_characteristics_4",
    "has_characteristics_5",
]

df["combination_pattern"] = df[category_cols].astype(str).agg("-".join, axis=1)

# Remove baseline category and rows without ring counts
df_filtered = df[
    (df["combination_pattern"] != "0-0-0-0-0") & (df["num_total_rings"].notnull())
]

target_size = 100000
pattern_counts = df_filtered["combination_pattern"].value_counts().to_dict()

# First water-fill: allocate target size across combination patterns
pattern_allocations = calculate_water_fill(pattern_counts, target_size)

sampled_list = []

for pattern, pattern_target in pattern_allocations.items():
    if pattern_target > 0:
        group_df = df_filtered[df_filtered["combination_pattern"] == pattern]

        ring_counts = group_df["num_total_rings"].value_counts().to_dict()

        # Second water-fill: allocate pattern target size across its ring sizes
        ring_allocations = calculate_water_fill(ring_counts, pattern_target)

        for ring_size, ring_target in ring_allocations.items():
            if ring_target > 0:
                ring_group_df = group_df[group_df["num_total_rings"] == ring_size]
                sampled_group = ring_group_df.sample(n=ring_target, random_state=42)
                sampled_list.append(sampled_group)

sampled_df = (
    pd.concat(sampled_list).sample(frac=1, random_state=42).reset_index(drop=True)
)

cursor = conn.cursor()
try:
    cursor.execute("ALTER TABLE pac5 ADD COLUMN selected INTEGER DEFAULT 0")
except sqlite3.OperationalError as e:
    if "duplicate column name" in str(e):
        print("Column 'selected' already exists. Resetting all to 0.")
        cursor.execute("UPDATE pac5 SET selected = 0")
    else:
        raise e

print(f"Updating {len(sampled_df)} rows")
selected_ids = [(int(cid),) for cid in sampled_df["pubchem_cid"]]
cursor.executemany(
    """
    UPDATE pac5
    SET selected = 1
    WHERE pubchem_cid = ?
""",
    selected_ids,
)

conn.commit()
conn.close()

print(f"Molecules selected: {len(sampled_df)}")

Column 'selected' already exists. Resetting all to 0.
Updating 100000 rows
Molecules selected: 100000


In [6]:
import sqlite3

import pandas as pd


def calculate_water_fill(counts_dict, target_size):
    available_keys = list(counts_dict.keys())
    allocated = {k: 0 for k in available_keys}
    remaining_target = target_size

    while remaining_target > 0 and len(available_keys) > 0:
        per_key_target = remaining_target // len(available_keys)
        shortfall_keys = [k for k in available_keys if counts_dict[k] <= per_key_target]

        if shortfall_keys:
            for k in shortfall_keys:
                allocated[k] = counts_dict[k]
                remaining_target -= counts_dict[k]
                available_keys.remove(k)
        else:
            for k in available_keys:
                allocated[k] = per_key_target
                remaining_target -= per_key_target

            for k in available_keys[:remaining_target]:
                allocated[k] += 1
            remaining_target = 0
            break

    return allocated


db_path = "../data/pac5.db"
conn = sqlite3.connect(db_path)

query = """
SELECT
    pubchem_cid,
    has_characteristics_1,
    has_characteristics_2,
    has_characteristics_3,
    has_characteristics_4,
    has_characteristics_5,
    num_total_rings
FROM pac5 WHERE excluded = 0 AND num_atoms > 12 AND num_total_rings > 1
"""
df = pd.read_sql_query(query, conn)

category_cols = [
    "has_characteristics_1",
    "has_characteristics_2",
    "has_characteristics_3",
    "has_characteristics_4",
    "has_characteristics_5",
]

df["combination_pattern"] = df[category_cols].astype(str).agg("-".join, axis=1)


print("Sub-sampling 2,000 high-priority molecules from the selected 100,000...")
target_size_high = 2000
pattern_counts_high = df["combination_pattern"].value_counts().to_dict()
pattern_allocations_high = calculate_water_fill(pattern_counts_high, target_size_high)

sampled_high_list = []

for pattern, pattern_target in pattern_allocations_high.items():
    if pattern_target > 0:
        group_df_high = df[df["combination_pattern"] == pattern]
        ring_counts_high = group_df_high["num_total_rings"].value_counts().to_dict()

        # Third water-fill: allocate pattern high-priority target size across its ring sizes
        ring_allocations_high = calculate_water_fill(ring_counts_high, pattern_target)

        for ring_size, ring_target in ring_allocations_high.items():
            if ring_target > 0:
                ring_group_df_high = group_df_high[
                    group_df_high["num_total_rings"] == ring_size
                ]
                sampled_group_high = ring_group_df_high.sample(
                    n=ring_target, random_state=42
                )
                sampled_high_list.append(sampled_group_high)

sampled_high_df = (
    pd.concat(sampled_high_list).sample(frac=1, random_state=42).reset_index(drop=True)
)

cursor = conn.cursor()

try:
    cursor.execute("ALTER TABLE pac5 ADD COLUMN camb3lyp_selected INTEGER DEFAULT 0")
except sqlite3.OperationalError as e:
    if "duplicate column name" in str(e):
        print("Column 'camb3lyp_selected' already exists. Resetting all to 0.")
        cursor.execute("UPDATE pac5 SET camb3lyp_selected = 0")
    else:
        raise e

print(f"Updating {len(sampled_high_df)} rows")
camb3lyp_selected_ids = [(int(cid),) for cid in sampled_high_df["pubchem_cid"]]
cursor.executemany(
    """
    UPDATE pac5
    SET camb3lyp_selected = 1
    WHERE pubchem_cid = ?
""",
    camb3lyp_selected_ids,
)

conn.commit()
conn.close()

print(f"High priority selected: {len(sampled_high_df)}")

Sub-sampling 2,000 high-priority molecules from the selected 100,000...
Column 'camb3lyp_selected' already exists. Resetting all to 0.
Updating 2000 rows
High priority selected: 2000


In [1]:
import sqlite3

import pandas as pd


def calculate_water_fill(counts_dict, target_size):
    available_keys = list(counts_dict.keys())
    allocated = {k: 0 for k in available_keys}
    remaining_target = target_size

    while remaining_target > 0 and len(available_keys) > 0:
        per_key_target = remaining_target // len(available_keys)
        shortfall_keys = [k for k in available_keys if counts_dict[k] <= per_key_target]

        if shortfall_keys:
            for k in shortfall_keys:
                allocated[k] = counts_dict[k]
                remaining_target -= counts_dict[k]
                available_keys.remove(k)
        else:
            for k in available_keys:
                allocated[k] = per_key_target
                remaining_target -= per_key_target

            for k in available_keys[:remaining_target]:
                allocated[k] += 1
            remaining_target = 0
            break

    return allocated


db_path = "../data/pac5.db"
conn = sqlite3.connect(db_path)

query = """
SELECT
    pubchem_cid,
    has_characteristics_1,
    has_characteristics_2,
    has_characteristics_3,
    has_characteristics_4,
    has_characteristics_5,
    num_total_rings,
    num_fg_nitro, num_fg_cyanide, num_fg_dimethylamino,
    num_fg_methyl, num_fg_ethyl, num_fg_fluoride,
    num_fg_chloride, num_fg_carbonyl, num_fg_hydrogen
FROM pac5 WHERE excluded = 0 AND num_atoms > 12 AND num_total_rings > 1 ORDER BY pubchem_cid ASC
"""
df = pd.read_sql_query(query, conn)

category_cols = [
    "has_characteristics_1",
    "has_characteristics_2",
    "has_characteristics_3",
    "has_characteristics_4",
    "has_characteristics_5",
]
df["combination_pattern"] = df[category_cols].astype(str).agg("-".join, axis=1)

fg_cols = [
    "num_fg_nitro",
    "num_fg_cyanide",
    "num_fg_dimethylamino",
    "num_fg_methyl",
    "num_fg_ethyl",
    "num_fg_fluoride",
    "num_fg_chloride",
    "num_fg_carbonyl",
    "num_fg_hydrogen",
]

# Create binary presence columns for rarity calculation
bin_cols = []
for col in fg_cols:
    bin_name = col + "_bin"
    df[bin_name] = (df[col] > 0).astype(int)
    bin_cols.append(bin_name)

# Sort functional groups from rarest to most common globally
fg_rarity = df[bin_cols].sum().sort_values().index.tolist()

# Assign 'primary_fg' as rarest functional group present in molecule
conditions = [df[col] == 1 for col in fg_rarity]
choices = [col.replace("_bin", "") for col in fg_rarity]
df["primary_fg"] = np.select(conditions, choices, default="None")


target_size_high = 2000
pattern_counts_high = df["combination_pattern"].value_counts().to_dict()
pattern_allocations_high = calculate_water_fill(pattern_counts_high, target_size_high)

sampled_high_list = []

# Tier 1: Categories
for pattern, pattern_target in pattern_allocations_high.items():
    if pattern_target > 0:
        group_df_high = df[df["combination_pattern"] == pattern]
        ring_counts_high = group_df_high["num_total_rings"].value_counts().to_dict()

        # Tier 2: Number of Rings
        ring_allocations_high = calculate_water_fill(ring_counts_high, pattern_target)

        for ring_size, ring_target in ring_allocations_high.items():
            if ring_target > 0:
                ring_group_df_high = group_df_high[
                    group_df_high["num_total_rings"] == ring_size
                ]

                fg_counts_high = (
                    ring_group_df_high["primary_fg"].value_counts().to_dict()
                )

                # Tier 3: Primary Functional Group
                fg_allocations_high = calculate_water_fill(fg_counts_high, ring_target)

                for fg_type, fg_target in fg_allocations_high.items():
                    if fg_target > 0:
                        fg_group_df = ring_group_df_high[
                            ring_group_df_high["primary_fg"] == fg_type
                        ]
                        sampled_group = fg_group_df.sample(n=fg_target, random_state=42)
                        sampled_high_list.append(sampled_group)

sampled_high_df = (
    pd.concat(sampled_high_list).sample(frac=1, random_state=42).reset_index(drop=True)
)

cursor = conn.cursor()

try:
    cursor.execute("ALTER TABLE pac5 ADD COLUMN camb3lyp_selected INTEGER DEFAULT 0")
except sqlite3.OperationalError as e:
    if "duplicate column name" in str(e):
        print("Column 'camb3lyp_selected' already exists. Resetting all to 0.")
        cursor.execute("UPDATE pac5 SET camb3lyp_selected = 0")
    else:
        raise e

print(f"Updating {len(sampled_high_df)} rows")
camb3lyp_selected_ids = [(int(cid),) for cid in sampled_high_df["pubchem_cid"]]
cursor.executemany(
    """
    UPDATE pac5
    SET camb3lyp_selected = 1
    WHERE pubchem_cid = ?
""",
    camb3lyp_selected_ids,
)

conn.commit()
conn.close()

print(f"High priority selected: {len(sampled_high_df)}")

Column 'camb3lyp_selected' already exists. Resetting all to 0.
Updating 2000 rows
High priority selected: 2000


In [2]:
import sqlite3

import pandas as pd
import plotly.express as px

db_path = "../data/pac5.db"
conn = sqlite3.connect(db_path)

query = """
SELECT
    has_characteristics_1,
    has_characteristics_2,
    has_characteristics_3,
    has_characteristics_4,
    has_characteristics_5,
    num_total_rings
FROM pac5 WHERE camb3lyp_selected = 1 AND num_total_rings > 1
"""
df = pd.read_sql_query(query, conn)
conn.close()

category_cols = [
    "has_characteristics_1",
    "has_characteristics_2",
    "has_characteristics_3",
    "has_characteristics_4",
    "has_characteristics_5",
]

df["combination_pattern"] = df[category_cols].astype(str).agg("-".join, axis=1)

df_filtered = df[df["combination_pattern"] != "0-0-0-0-0"].copy()

df_filtered["num_total_rings"] = (
    df_filtered["num_total_rings"].fillna(-1).astype(int).astype(str)
)

pattern_totals = df_filtered["combination_pattern"].value_counts().index.tolist()

combination_counts = (
    df_filtered.groupby(["combination_pattern", "num_total_rings"])
    .size()
    .reset_index(name="Molecule count")
)

fig = px.bar(
    combination_counts,
    x="combination_pattern",
    y="Molecule count",
    color="num_total_rings",
    title="Frequency of category combinations for PACs by ring count",
    labels={
        "combination_pattern": "Binary state (1=Present, 0=Absent)",
        "num_total_rings": "Total rings",
    },
    log_y=False,
    template="plotly_white",
    category_orders={"combination_pattern": pattern_totals},
)

fig.update_layout(
    xaxis_tickangle=-45, margin=dict(t=50, b=50, l=50, r=50), barmode="stack"
)

fig.show()

In [3]:
import sqlite3

import pandas as pd
import plotly.express as px

db_path = "../data/pac5.db"
conn = sqlite3.connect(db_path)

query = """
SELECT
    num_fg_nitro,
    num_fg_cyanide,
    num_fg_dimethylamino,
    num_fg_methyl,
    num_fg_ethyl,
    num_fg_fluoride,
    num_fg_chloride,
    num_fg_carbonyl,
    num_fg_hydrogen
FROM pac5 WHERE camb3lyp_selected = 1 AND num_total_rings > 1
"""
df = pd.read_sql_query(query, conn)
conn.close()

df_binary = (df > 0).astype(int)

fg_counts = df_binary.sum().reset_index()
fg_counts.columns = ["Functional Group", "Molecule Count"]

fg_counts["Functional Group"] = (
    fg_counts["Functional Group"].str.replace("num_fg_", "").str.capitalize()
)
fg_counts = fg_counts.sort_values("Molecule Count", ascending=False)

fig = px.bar(
    fg_counts,
    x="Functional Group",
    y="Molecule Count",
    title="Presence of Functional Groups in Selected PACs (Binary)",
    text_auto=True,
    template="plotly_white",
)

fig.update_layout(
    xaxis_title="Functional Group",
    yaxis_title="Number of Molecules Containing the Group",
    margin=dict(t=50, b=50, l=50, r=50),
)

fig.show()

In [4]:
import sqlite3

import pandas as pd
import plotly.express as px

db_path = "../data/pac5.db"
conn = sqlite3.connect(db_path)

query = """
SELECT
    num_heterocyclic_nitrogen,
    num_heterocyclic_oxygen,
    num_heterocyclic_sulfur,
    num_sp3_carbon
FROM pac5 WHERE camb3lyp_selected = 1 AND num_total_rings > 1
"""
df = pd.read_sql_query(query, conn)
conn.close()

df_binary = (df > 0).astype(int)

atom_counts = df_binary.sum().reset_index()
atom_counts.columns = ["Atom Type", "Molecule Count"]

atom_counts["Atom Type"] = (
    atom_counts["Atom Type"]
    .str.replace("num_heterocyclic_", "Heterocyclic ")
    .str.replace("num_", "")
    .str.replace("_", " ")
    .str.title()
)
atom_counts = atom_counts.sort_values("Molecule Count", ascending=False)

fig = px.bar(
    atom_counts,
    x="Atom Type",
    y="Molecule Count",
    title="Presence of Specific Atom Types in Selected PACs (Binary)",
    text_auto=True,
    template="plotly_white",
)

fig.update_layout(
    xaxis_title="Atom/Structural Type",
    yaxis_title="Number of Molecules Containing the Type",
    margin=dict(t=50, b=50, l=50, r=50),
)

fig.show()